# PanDx Reproduction Notebook
**AI-assisted Early Detection of Pancreatic Ductal Adenocarcinoma on Contrast-enhanced CT**

Liu et al. (2025) — 1st place, PANORAMA Challenge
- Paper: https://arxiv.org/abs/2503.10068
- Code:  https://github.com/han-liu/PanDx

## Pipeline overview
```
CT scan (full resolution)
   │
   ▼
[Dataset Prep]  Data exploration → DASE split → nnU-Net format conversion
   │
   ▼
[Stage 1]  Downsample → nnU-Net (Dataset103) → Pancreas segmentation mask
   │
   ▼
[Stage 2]  Crop ROI (±100/50/15 mm) → nnU-Net (Dataset107, CE loss) → PDAC prob map
   │
   ▼
[Post]     Peak-scaled candidate extraction (α = 1/15) → patient likelihood score
```

## PANORAMA label map
| Label | Structure |
|---|---|
| 0 | Background |
| 1 | Pancreas parenchyma |
| 2 | PDAC lesion |
| 3 | Pancreatic duct |
| 4 | Common bile duct |
| 5 | Veins |
| 6 | Arteries |

## 0. Environment setup & imports

In [ ]:
import os
import os.path as osp
import json
import shutil
import subprocess
import hashlib
import time
import warnings
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Add local packages to path
import sys
REPO_ROOT = osp.abspath('.')          # /Desktop/PanDx
sys.path.insert(0, osp.join(REPO_ROOT, 'packages', 'nnunetv2'))
sys.path.insert(0, osp.join(REPO_ROOT, 'packages', 'report-guided-annotation', 'src'))

from report_guided_annotation.extract_lesion_candidates import extract_lesion_candidates

print('Imports OK')
print('REPO_ROOT:', REPO_ROOT)

## 1. Configuration — set your paths here

In [ ]:
# ── Raw PANORAMA dataset ───────────────────────────────────────────────────────
# Folder containing all CT scans (.mha or .nii.gz)
IMAGE_DIR = osp.join(REPO_ROOT, 'workspace', 'raw_data', 'images')
# Folder containing corresponding segmentation masks (same filenames)
LABEL_DIR = osp.join(REPO_ROOT, 'workspace', 'raw_data', 'labels')

# ── Inference I/O paths ───────────────────────────────────────────────────────
# For single-case testing use the test_example folder;
# for the full dataset point INPUT_DIR at IMAGE_DIR above
INPUT_DIR   = osp.join(REPO_ROOT, 'workspace', 'test_example', 'input')
OUTPUT_DIR  = osp.join(REPO_ROOT, 'workspace', 'test_example', 'output')
MODEL_DIR   = osp.join(REPO_ROOT, 'workspace', 'nnUNet_results')
WORKING_DIR = osp.join(OUTPUT_DIR, 'itm')

# ── nnU-Net dataset paths ─────────────────────────────────────────────────────
NNUNET_RAW          = osp.join(REPO_ROOT, 'workspace', 'nnUNet_raw')
NNUNET_PREPROCESSED = osp.join(REPO_ROOT, 'workspace', 'nnUNet_preprocessed')
DATASET_NAME        = 'Dataset107_PDAC_Detection'
DATASET_ID          = 107

# ── Model identifiers ─────────────────────────────────────────────────────────
STAGE1_TASK    = 103
STAGE1_TRAINER = 'nnUNetTrainer'
STAGE1_PLAN    = 'nnUNetPlans'

STAGE2_TASK    = 107
STAGE2_TRAINER = 'nnUNetTrainerCELossLesionSplit'
STAGE2_PLAN    = 'nnUNetPlans_v3'

# ── Hyper-parameters ──────────────────────────────────────────────────────────
DOWNSAMPLE_SPACING = (4.5, 4.5, 9.0)   # mm — low-res spacing for Stage 1
ROI_MARGINS        = [100, 50, 15]      # mm — x/y/z padding around pancreas bbox
INV_ALPHA          = 15                 # τ = (1/15) · P(x*)  peak-scaling factor
RANDOM_SEED        = 42                 # fixed seed for DASE split reproducibility
N_FOLDS            = 5

for d in [OUTPUT_DIR, WORKING_DIR, osp.join(OUTPUT_DIR, 'pdac-detection-map'),
          NNUNET_RAW, NNUNET_PREPROCESSED]:
    os.makedirs(d, exist_ok=True)

print('Config ready.')
print(f'  IMAGE_DIR : {IMAGE_DIR}')
print(f'  LABEL_DIR : {LABEL_DIR}')
print(f'  OUTPUT_DIR: {OUTPUT_DIR}')
print(f'  MODEL_DIR : {MODEL_DIR}')

## 2. Helper functions

In [ ]:
def get_file_extension(path: str) -> str:
    base, ext = osp.splitext(path)
    if ext == '.gz' and base.endswith('.nii'):
        return '.nii.gz'
    return ext


def resample_img(itk_image, out_spacing=(2.0, 2.0, 2.0), is_label=False,
                 out_size=None, out_origin=None, out_direction=None):
    """Resample an ITK image to the requested voxel spacing."""
    orig_spacing = itk_image.GetSpacing()
    orig_size    = itk_image.GetSize()
    if out_size is None:
        out_size = [
            int(np.round(orig_size[i] * orig_spacing[i] / out_spacing[i]))
            for i in range(3)
        ]
    rs = sitk.ResampleImageFilter()
    rs.SetOutputSpacing(out_spacing)
    rs.SetSize(out_size)
    rs.SetOutputDirection(out_direction or itk_image.GetDirection())
    rs.SetOutputOrigin(out_origin or itk_image.GetOrigin())
    rs.SetTransform(sitk.Transform())
    rs.SetDefaultPixelValue(itk_image.GetPixelIDValue())
    rs.SetInterpolator(sitk.sitkNearestNeighbor if is_label else sitk.sitkBSpline)
    return rs.Execute(itk_image)


def downsample_dataset(img_dir: str, save_dir: str,
                       spacing=DOWNSAMPLE_SPACING) -> None:
    """Downsample all images in img_dir to low-res spacing for Stage-1 inference."""
    os.makedirs(save_dir, exist_ok=True)
    img_paths = sorted(glob(osp.join(img_dir, '*.*')))
    assert img_paths, f'No images found in {img_dir}'
    for p in tqdm(img_paths, desc='Downsample'):
        ext  = get_file_extension(p)
        img  = sitk.ReadImage(p, sitk.sitkFloat32)
        resampled = resample_img(img, spacing)
        out_name  = osp.basename(p).replace(ext, '_0000.nii.gz')
        sitk.WriteImage(resampled, osp.join(save_dir, out_name))


def crop_roi(img_dir: str, low_mask_dir: str, save_dir: str,
             margins=ROI_MARGINS) -> dict:
    """
    Crop the high-resolution CT around the predicted pancreas bounding box.
    Returns a dict mapping case-id → crop coordinate slices.
    """
    os.makedirs(save_dir, exist_ok=True)
    img_paths = sorted(glob(osp.join(img_dir, '*.*')))
    crop_coords = {}
    for p in tqdm(img_paths, desc='Crop ROI'):
        ext       = get_file_extension(p)
        mask_path = osp.join(low_mask_dir, osp.basename(p).replace(ext, '.nii.gz'))
        img       = sitk.ReadImage(p, sitk.sitkFloat32)
        low_mask  = sitk.ReadImage(mask_path)
        mask_np   = sitk.GetArrayFromImage(low_mask)
        mask_np   = (mask_np == 1).astype(np.uint8)
        nz = np.nonzero(mask_np)
        min_x, max_x = int(nz[2].min()), int(nz[2].max())
        min_y, max_y = int(nz[1].min()), int(nz[1].max())
        min_z, max_z = int(nz[0].min()), int(nz[0].max())
        sp = low_mask.TransformIndexToPhysicalPoint
        ip = img.TransformPhysicalPointToIndex
        start_idx  = ip(sp((min_x, min_y, min_z)))
        finish_idx = ip(sp((max_x, max_y, max_z)))
        spacing = img.GetSpacing()
        size    = img.GetSize()
        mx = int(margins[0] / spacing[0])
        my = int(margins[1] / spacing[1])
        mz = int(margins[2] / spacing[2])
        xs = max(0, start_idx[0] - mx);  xf = min(size[0], finish_idx[0] + mx)
        ys = max(0, start_idx[1] - my);  yf = min(size[1], finish_idx[1] + my)
        zs = max(0, start_idx[2] - mz);  zf = min(size[2], finish_idx[2] + mz)
        cropped = img[xs:xf, ys:yf, zs:zf]
        case_id = osp.basename(p).replace(ext, '')
        crop_coords[case_id] = dict(x_start=xs, x_finish=xf,
                                    y_start=ys, y_finish=yf,
                                    z_start=zs, z_finish=zf)
        sitk.WriteImage(cropped,
                        osp.join(save_dir, osp.basename(p).replace(ext, '_0000.nii.gz')))
    return crop_coords


def run_nnunet_predict(model_dir: str, input_dir: str, output_dir: str,
                       task: int, trainer: str = 'nnUNetTrainer',
                       plan: str = 'nnUNetPlans',
                       configuration: str = '3d_fullres',
                       checkpoint: str = 'checkpoint_final.pth',
                       folds: str = '0,1,2,3,4',
                       save_probs: bool = True,
                       tta: bool = True) -> None:
    """Call nnUNetv2_predict as a subprocess."""
    os.environ['RESULTS_FOLDER'] = model_dir
    os.makedirs(output_dir, exist_ok=True)
    cmd = [
        'nnUNetv2_predict',
        '-d',  str(task),
        '-i',  input_dir,
        '-o',  output_dir,
        '-c',  configuration,
        '-tr', trainer,
        '-p',  plan,
        '--continue_prediction',
        '-f',  *folds.split(','),
        '-chk', checkpoint,
    ]
    if save_probs:
        cmd.append('--save_probabilities')
    if not tta:
        cmd.append('--disable_tta')
    print('Running:', ' '.join(str(c) for c in cmd))
    subprocess.check_call(cmd)


def postprocess_probmap(npz_path: str) -> np.ndarray:
    """Load .npz from nnU-Net and extract the PDAC (class 1) probability map."""
    data = np.load(npz_path)
    return data['probabilities'][1].astype(np.float32)


def build_full_size_detection_map(prob_map: np.ndarray,
                                  crop_coords: dict,
                                  reference_image: sitk.Image,
                                  inv_alpha: int = INV_ALPHA):
    """
    Apply peak-scaled lesion candidate extraction (τ = prob_map.max() / inv_alpha),
    then paste the result back into a full-size volume.
    Returns (detection_map_itk, patient_likelihood_score).
    """
    lesion_candidates, _, _ = extract_lesion_candidates(
        prob_map, dynamic_threshold_factor=inv_alpha
    )
    patient_score = float(np.max(lesion_candidates))
    full_shape = sitk.GetArrayFromImage(reference_image).shape
    full_map   = np.zeros(full_shape, dtype=np.float32)
    full_map[
        crop_coords['z_start']:crop_coords['z_finish'],
        crop_coords['y_start']:crop_coords['y_finish'],
        crop_coords['x_start']:crop_coords['x_finish'],
    ] = lesion_candidates
    out_itk = sitk.GetImageFromArray(full_map)
    out_itk.CopyInformation(reference_image)
    return out_itk, patient_score


def write_json(path: str, content: dict) -> None:
    with open(path, 'w') as f:
        json.dump(content, f, indent=4)


print('Helper functions defined.')

---
# Part A — Dataset Preparation
---

## 3. Dataset inspection
Summarise what images are in `IMAGE_DIR` and display one representative slice.

In [ ]:
img_paths = sorted([
    p for p in glob(osp.join(IMAGE_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])
print(f'Found {len(img_paths)} image(s) in {IMAGE_DIR}:')
for p in img_paths[:10]:   # show first 10 to avoid flooding output
    img = sitk.ReadImage(p)
    print(f'  {osp.basename(p):45s}  size={img.GetSize()}  '
          f'spacing={tuple(f"{s:.2f}" for s in img.GetSpacing())}')
if len(img_paths) > 10:
    print(f'  ... and {len(img_paths) - 10} more')

In [ ]:
# Visualise a central axial slice of the first image
if img_paths:
    ex    = sitk.ReadImage(img_paths[0], sitk.sitkFloat32)
    arr   = sitk.GetArrayFromImage(ex)
    mid_z = arr.shape[0] // 2
    plt.figure(figsize=(6, 6))
    plt.imshow(arr[mid_z], cmap='gray', vmin=-200, vmax=300)
    plt.title(f'{osp.basename(img_paths[0])} — axial slice {mid_z}')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Data exploration

Compute for every case:
- Whether it is PDAC-positive (label 2 present in mask)
- PDAC lesion volume in mm³
- Age and sex from the clinical metadata JSON (if available)

**Expected finding (Fig. 1 of the paper):** lesion size is heavily right-skewed — most lesions are small (< 1 000 mm³).

In [ ]:
label_paths = sorted([
    p for p in glob(osp.join(LABEL_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])

cases = []
for label_path in tqdm(label_paths, desc='Scanning labels'):
    ext     = get_file_extension(label_path)
    case_id = osp.basename(label_path).replace(ext, '')

    mask    = sitk.ReadImage(label_path)
    arr     = sitk.GetArrayFromImage(mask)
    spacing = mask.GetSpacing()                        # (x, y, z) in mm
    vox_vol = spacing[0] * spacing[1] * spacing[2]    # mm³ per voxel

    pdac_voxels  = int((arr == 2).sum())
    pdac_vol_mm3 = pdac_voxels * vox_vol

    entry = {
        'case_id':      case_id,
        'is_pdac':      pdac_voxels > 0,
        'pdac_vol_mm3': pdac_vol_mm3,
        'age':          None,
        'sex':          None,
    }

    # Load clinical metadata JSON (one per case, same stem as image)
    for json_dir in [IMAGE_DIR, LABEL_DIR]:
        json_path = osp.join(json_dir, f'{case_id}.json')
        if osp.exists(json_path):
            with open(json_path) as f:
                meta = json.load(f)
            entry['age'] = meta.get('age')
            entry['sex'] = meta.get('sex')
            break

    cases.append(entry)

df = pd.DataFrame(cases)
print(f'Total cases   : {len(df)}')
print(f'PDAC positive : {df.is_pdac.sum()}')
print(f'PDAC negative : {(~df.is_pdac).sum()}')
print(f'\nLesion volume stats (PDAC-positive cases only):')
print(df[df.is_pdac]['pdac_vol_mm3'].describe().round(1))

In [ ]:
# ── Lesion size distribution ──────────────────────────────────────────────────
pdac_vols = df[df.is_pdac]['pdac_vol_mm3']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(pdac_vols, bins=50, color='tomato', edgecolor='white')
axes[0].set_xlabel('Lesion volume (mm³)')
axes[0].set_ylabel('Number of cases')
axes[0].set_title('Lesion size distribution — linear scale\n(note strong right skew toward small lesions)')

axes[1].hist(np.log1p(pdac_vols), bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log(1 + lesion volume)')
axes[1].set_ylabel('Number of cases')
axes[1].set_title('Lesion size distribution — log scale')

# Mark quartile boundaries on the log-scale plot
for q, c in zip([0.25, 0.50, 0.75], ['gold', 'orange', 'red']):
    val = np.log1p(pdac_vols.quantile(q))
    axes[1].axvline(val, color=c, linestyle='--', label=f'Q{int(q*4)} boundary')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Age / sex distribution ────────────────────────────────────────────────────
has_meta = df['age'].notna().any()

if has_meta:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Age histogram split by PDAC status
    for label, colour in [('PDAC+', 'tomato'), ('PDAC-', 'steelblue')]:
        is_p = label == 'PDAC+'
        ages = df[df.is_pdac == is_p]['age'].dropna()
        axes[0].hist(ages, bins=20, alpha=0.6, color=colour, label=label, edgecolor='white')
    axes[0].set_xlabel('Age (years)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Age distribution')
    axes[0].legend()

    # Sex breakdown
    sex_counts = df.groupby(['sex', 'is_pdac']).size().unstack(fill_value=0)
    sex_counts.columns = ['PDAC-', 'PDAC+']
    sex_counts.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'],
                    edgecolor='white', rot=0)
    axes[1].set_title('Sex distribution')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()
else:
    print('No clinical metadata found — skipping age/sex plots.')
    print('(Expected JSON files alongside images with "age" and "sex" keys.)')

## 5. DASE split — Distribution-Aware Stratified Evaluation

**Goal:** ensure every fold gets a proportional share of both PDAC+/- cases *and* all lesion size ranges.

**Algorithm:**
1. Sort PDAC+ cases by lesion volume → divide into 4 quartile bins (Q1–Q4)
2. Within each bin, assign cases round-robin across 5 folds
3. Distribute PDAC- cases round-robin to preserve the global PDAC+/- ratio
4. Fix `RANDOM_SEED = 42` for full reproducibility

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

pdac_pos = df[df.is_pdac].copy().sort_values('pdac_vol_mm3').reset_index(drop=True)
pdac_neg = df[~df.is_pdac].copy().reset_index(drop=True)

# Assign quartile size bin to each PDAC+ case
pdac_pos['size_bin'] = pd.qcut(
    pdac_pos['pdac_vol_mm3'],
    q=4,
    labels=['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)']
)

print('PDAC+ cases per size quartile:')
print(pdac_pos['size_bin'].value_counts().sort_index().to_string())
print()

# Round-robin assignment within each quartile
fold_assignments = {}
for bin_label in ['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)']:
    bin_cases = pdac_pos[pdac_pos.size_bin == bin_label]['case_id'].tolist()
    rng.shuffle(bin_cases)
    for i, case_id in enumerate(bin_cases):
        fold_assignments[case_id] = i % N_FOLDS

# Round-robin assignment for PDAC- cases
neg_cases = pdac_neg['case_id'].tolist()
rng.shuffle(neg_cases)
for i, case_id in enumerate(neg_cases):
    fold_assignments[case_id] = i % N_FOLDS

df['fold'] = df['case_id'].map(fold_assignments)

# ── Summary table ─────────────────────────────────────────────────────────────
print('Fold composition:')
for fold in range(N_FOLDS):
    fold_df = df[df.fold == fold]
    n_pos   = fold_df.is_pdac.sum()
    n_neg   = (~fold_df.is_pdac).sum()
    ratio   = n_pos / len(fold_df) * 100
    print(f'  Fold {fold}: {len(fold_df):4d} cases | '
          f'PDAC+ = {n_pos:3d}  PDAC- = {n_neg:3d}  ({ratio:.1f}% positive)')

print()
print('PDAC+ lesion-size distribution per fold:')
print(
    pdac_pos
    .assign(fold=pdac_pos.case_id.map(fold_assignments))
    .groupby(['fold', 'size_bin'], observed=True)
    .size()
    .unstack(fill_value=0)
    .to_string()
)

In [ ]:
# ── Visualise the DASE split ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Stacked bar: PDAC+ (by quartile) and PDAC- counts per fold
fold_pos = (
    pdac_pos
    .assign(fold=pdac_pos.case_id.map(fold_assignments))
    .groupby(['fold', 'size_bin'], observed=True)
    .size()
    .unstack(fill_value=0)
)
fold_neg = df[~df.is_pdac].groupby('fold').size().rename('PDAC-')

colours = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
bottom  = np.zeros(N_FOLDS)
for col, colour in zip(fold_pos.columns, colours):
    axes[0].bar(fold_pos.index, fold_pos[col], bottom=bottom,
                color=colour, label=str(col), edgecolor='white')
    bottom += fold_pos[col].values
axes[0].bar(fold_neg.index, fold_neg.values, bottom=bottom,
            color='#9E9E9E', label='PDAC-', edgecolor='white')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Number of cases')
axes[0].set_title('DASE split — cases per fold by size quartile')
axes[0].legend(fontsize=8, loc='upper right')

# PDAC+ ratio per fold
ratio_per_fold = [df[df.fold == f].is_pdac.mean() * 100 for f in range(N_FOLDS)]
global_ratio   = df.is_pdac.mean() * 100
axes[1].bar(range(N_FOLDS), ratio_per_fold, color='tomato', edgecolor='white')
axes[1].axhline(global_ratio, color='black', linestyle='--', label=f'Global ratio ({global_ratio:.1f}%)')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('% PDAC-positive')
axes[1].set_title('PDAC+ ratio per fold (should be uniform)')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Save the split to CSV
split_csv = osp.join(REPO_ROOT, 'workspace', 'dase_split.csv')
df[['case_id', 'is_pdac', 'pdac_vol_mm3', 'fold']].to_csv(split_csv, index=False)
print(f'\nSplit saved to: {split_csv}')

## 6. Reproducibility check

Re-run this cell from scratch at any time.  
The **MD5 checksum must be identical** every run — if it changes you have a non-determinism bug.

In [ ]:
df_check  = pd.read_csv(split_csv)
fold_str  = df_check.sort_values('case_id')['fold'].astype(str).str.cat()
checksum  = hashlib.md5(fold_str.encode()).hexdigest()

print('=' * 50)
print(f'  Split checksum (MD5): {checksum}')
print('=' * 50)
print('Save this value. Re-running with RANDOM_SEED=42')
print('must always produce the same checksum.')
print()
print('Fold sizes:')
print(df_check.groupby('fold').size().rename('n_cases').to_string())

## 7. Convert dataset to nnU-Net format

nnU-Net requires:
```
nnUNet_raw/Dataset107_PDAC_Detection/
    dataset.json
    imagesTr/   ← images named  <case_id>_0000.nii.gz
    labelsTr/   ← labels named  <case_id>.nii.gz
```
Labels are kept as-is (multi-class 0–6) since nnU-Net accepts them directly.

In [ ]:
out_images = osp.join(NNUNET_RAW, DATASET_NAME, 'imagesTr')
out_labels = osp.join(NNUNET_RAW, DATASET_NAME, 'labelsTr')
os.makedirs(out_images, exist_ok=True)
os.makedirs(out_labels, exist_ok=True)

skipped = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc='Copying files'):
    cid = row['case_id']

    # Locate source image (try .nii.gz and .mha)
    src_img = None
    for ext in ['.nii.gz', '.mha', '.nii']:
        candidate = osp.join(IMAGE_DIR, f'{cid}{ext}')
        if osp.exists(candidate):
            src_img = candidate
            break
    if src_img is None:
        print(f'  WARNING: image not found for {cid} — skipping')
        skipped += 1
        continue

    # Locate source label
    src_lbl = None
    for ext in ['.nii.gz', '.mha', '.nii']:
        candidate = osp.join(LABEL_DIR, f'{cid}{ext}')
        if osp.exists(candidate):
            src_lbl = candidate
            break
    if src_lbl is None:
        print(f'  WARNING: label not found for {cid} — skipping')
        skipped += 1
        continue

    dst_img = osp.join(out_images, f'{cid}_0000.nii.gz')
    dst_lbl = osp.join(out_labels, f'{cid}.nii.gz')

    if not osp.exists(dst_img):
        # Convert .mha → .nii.gz if needed, otherwise plain copy
        if src_img.endswith('.nii.gz'):
            shutil.copy2(src_img, dst_img)
        else:
            img_itk = sitk.ReadImage(src_img, sitk.sitkFloat32)
            sitk.WriteImage(img_itk, dst_img)

    if not osp.exists(dst_lbl):
        if src_lbl.endswith('.nii.gz'):
            shutil.copy2(src_lbl, dst_lbl)
        else:
            lbl_itk = sitk.ReadImage(src_lbl)
            sitk.WriteImage(lbl_itk, dst_lbl)

n_copied = len(df) - skipped
print(f'\nCopied {n_copied} cases ({skipped} skipped).')

# Write dataset.json
dataset_json = {
    "channel_names": {"0": "CT"},
    "labels": {
        "background":          0,
        "pancreas":            1,
        "PDAC":                2,
        "pancreatic_duct":     3,
        "common_bile_duct":    4,
        "veins":               5,
        "arteries":            6
    },
    "numTraining": n_copied,
    "file_ending": ".nii.gz"
}
write_json(osp.join(NNUNET_RAW, DATASET_NAME, 'dataset.json'), dataset_json)
print(f'dataset.json written to {osp.join(NNUNET_RAW, DATASET_NAME)}')

## 8. nnU-Net plan & preprocess

This runs `nnUNetv2_plan_and_preprocess` **inside the notebook**.  
nnU-Net automatically handles: resampling, normalization, patch sizing, and architecture depth.

> **Tip:** If this cell takes > 10 minutes or the kernel crashes, run it in a terminal instead (copy the shell command printed below and paste it into your terminal).

In [ ]:
# Set required nnU-Net environment variables for this session
os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROCESSED
os.environ['nnUNet_results']      = MODEL_DIR

cmd = (
    f'nnUNetv2_plan_and_preprocess '
    f'-d {DATASET_ID} '
    f'--verify_dataset_integrity'
)
print('Command to run:')
print(f'  {cmd}')
print()
print('If you prefer the terminal, copy the command above and run it there.')
print('Otherwise, set RUN_PREPROCESS = True below and re-run this cell.')

In [ ]:
# Set to True only when ready — this step can take 30-60+ minutes on the full dataset
RUN_PREPROCESS = False

if RUN_PREPROCESS:
    print('Running nnU-Net plan & preprocess...')
    subprocess.check_call(cmd, shell=True)
    print('Done. Preprocessed files are in:', NNUNET_PREPROCESSED)
else:
    print('Skipped. Set RUN_PREPROCESS = True to run.')
    print('Or run in terminal:')
    print(f'  export nnUNet_raw="{NNUNET_RAW}"')
    print(f'  export nnUNet_preprocessed="{NNUNET_PREPROCESSED}"')
    print(f'  export nnUNet_results="{MODEL_DIR}"')
    print(f'  {cmd}')

---
# Part B — Inference Pipeline
---

## 9. Stage 1 — Pancreas localisation at low resolution

### 9a. Downsample to (4.5, 4.5, 9.0) mm

In [ ]:
LOW_IMAGE_DIR = osp.join(WORKING_DIR, 'LowImagesTr')
downsample_dataset(INPUT_DIR, LOW_IMAGE_DIR, DOWNSAMPLE_SPACING)
print('Downsampled images:', sorted(os.listdir(LOW_IMAGE_DIR)))

### 9b. nnU-Net inference — Dataset 103 (5-fold ensemble)

In [ ]:
LOW_PRED_DIR = osp.join(WORKING_DIR, 'LowPred')

run_nnunet_predict(
    model_dir  = MODEL_DIR,
    input_dir  = LOW_IMAGE_DIR,
    output_dir = LOW_PRED_DIR,
    task       = STAGE1_TASK,
    trainer    = STAGE1_TRAINER,
    plan       = STAGE1_PLAN,
    folds      = '0,1,2,3,4',
    save_probs = True,
    tta        = True,
)
print('Stage-1 predictions:', sorted(os.listdir(LOW_PRED_DIR)))

### 9c. Visualise Stage-1 segmentation overlay

In [ ]:
low_preds = sorted(glob(osp.join(LOW_PRED_DIR, '*.nii.gz')))
if low_preds:
    low_img_path  = sorted(glob(osp.join(LOW_IMAGE_DIR, '*.nii.gz')))[0]
    low_mask_path = low_preds[0]
    low_arr  = sitk.GetArrayFromImage(sitk.ReadImage(low_img_path, sitk.sitkFloat32))
    mask_arr = sitk.GetArrayFromImage(sitk.ReadImage(low_mask_path))
    best_z   = int(np.argmax((mask_arr == 1).sum(axis=(1, 2))))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(low_arr[best_z], cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title(f'Low-res CT — slice {best_z}')
    axes[0].axis('off')

    axes[1].imshow(low_arr[best_z],         cmap='gray',   vmin=-200, vmax=300)
    axes[1].imshow(mask_arr[best_z] == 1,   cmap='Reds',   alpha=0.4)
    axes[1].imshow(mask_arr[best_z] == 2,   cmap='Greens', alpha=0.5)
    axes[1].imshow(mask_arr[best_z] == 3,   cmap='Blues',  alpha=0.4)
    axes[1].set_title('Stage-1 segmentation overlay')
    axes[1].axis('off')

    patches = [
        mpatches.Patch(color='red',   alpha=0.6, label='Pancreas (1)'),
        mpatches.Patch(color='green', alpha=0.6, label='PDAC (2)'),
        mpatches.Patch(color='blue',  alpha=0.6, label='Duct (3)'),
    ]
    axes[1].legend(handles=patches, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

## 10. Stage 1 → Stage 2: Crop high-resolution ROI

Expand the predicted pancreas bounding box by **100 × 50 × 15 mm³** in x/y/z.

In [ ]:
CROPPED_IMAGE_DIR = osp.join(WORKING_DIR, 'CroppedImages')

crop_coordinates = crop_roi(
    img_dir      = INPUT_DIR,
    low_mask_dir = LOW_PRED_DIR,
    save_dir     = CROPPED_IMAGE_DIR,
    margins      = ROI_MARGINS,
)

print('\nCrop coordinates:')
for case_id, coords in crop_coordinates.items():
    print(f'  {case_id}: {coords}')

In [ ]:
# Visualise one cropped ROI vs the full scan
infer_img_paths = sorted([
    p for p in glob(osp.join(INPUT_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])
if infer_img_paths:
    full_arr    = sitk.GetArrayFromImage(sitk.ReadImage(infer_img_paths[0], sitk.sitkFloat32))
    cropped_arr = sitk.GetArrayFromImage(
        sitk.ReadImage(sorted(glob(osp.join(CROPPED_IMAGE_DIR, '*.nii.gz')))[0],
                       sitk.sitkFloat32)
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(full_arr[full_arr.shape[0]//2],       cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title('Full CT (mid-axial)')
    axes[0].axis('off')
    axes[1].imshow(cropped_arr[cropped_arr.shape[0]//2], cmap='gray', vmin=-200, vmax=300)
    axes[1].set_title('Cropped ROI (mid-axial)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 11. Stage 2 — Fine-scale PDAC detection

nnU-Net with **ResU-Net backbone** and **CE-only loss** (`nnUNetTrainerCELossLesionSplit`),  
trained on Dataset 107 (cropped pancreas ROIs).  
Softmax outputs are automatically averaged across all 5 folds during inference.

In [ ]:
CROPPED_PRED_DIR = osp.join(WORKING_DIR, 'CroppedPred')

run_nnunet_predict(
    model_dir  = MODEL_DIR,
    input_dir  = CROPPED_IMAGE_DIR,
    output_dir = CROPPED_PRED_DIR,
    task       = STAGE2_TASK,
    trainer    = STAGE2_TRAINER,
    plan       = STAGE2_PLAN,
    folds      = '0,1,2,3,4',
    save_probs = True,
    tta        = True,
)
print('Stage-2 predictions:', sorted(os.listdir(CROPPED_PRED_DIR)))

## 12. Post-processing — peak-scaled lesion candidate extraction

**Algorithm (from the paper):**
1. Set threshold τ = (1 / `inv_alpha`) × P(x*) where x* is the max-probability voxel.
2. Keep connected components above τ → lesion candidates.
3. Patient-level likelihood = max voxel probability in the candidate map.
4. Paste back into a full-size volume aligned to the original CT.

In [ ]:
npz_fps = sorted(glob(osp.join(CROPPED_PRED_DIR, '*.npz')))
infer_img_paths = sorted([
    p for p in glob(osp.join(INPUT_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])

assert len(npz_fps) == len(infer_img_paths), (
    f'Mismatch: {len(npz_fps)} predictions vs {len(infer_img_paths)} input images'
)

likelihoods = {}
for npz_fp, img_fp in zip(npz_fps, infer_img_paths):
    ext     = get_file_extension(img_fp)
    case_id = osp.basename(npz_fp)[:-4]
    assert osp.basename(img_fp).replace(ext, '') == case_id, \
        f'File order mismatch: {osp.basename(img_fp)} vs {osp.basename(npz_fp)}'

    ref_img  = sitk.ReadImage(img_fp, sitk.sitkFloat32)
    prob_map = postprocess_probmap(npz_fp)
    det_map, score = build_full_size_detection_map(
        prob_map, crop_coordinates[case_id], ref_img, INV_ALPHA
    )
    out_path = osp.join(OUTPUT_DIR, 'pdac-detection-map', f'{case_id}.nii.gz')
    sitk.WriteImage(det_map, out_path)
    likelihoods[case_id] = score
    print(f'  {case_id:40s}  likelihood = {score:.4f}')

write_json(osp.join(OUTPUT_DIR, 'pdac-likelihood.json'), likelihoods)
print(f'\nLikelihood scores → {osp.join(OUTPUT_DIR, "pdac-likelihood.json")}')

## 13. Visualise detection maps

In [ ]:
det_map_paths = sorted(glob(osp.join(OUTPUT_DIR, 'pdac-detection-map', '*.nii.gz')))

for det_path, img_fp in zip(det_map_paths, infer_img_paths):
    ct_arr  = sitk.GetArrayFromImage(sitk.ReadImage(img_fp,   sitk.sitkFloat32))
    det_arr = sitk.GetArrayFromImage(sitk.ReadImage(det_path, sitk.sitkFloat32))
    case_id = osp.basename(det_path).replace('.nii.gz', '')
    best_z  = int(np.argmax(det_arr.max(axis=(1, 2))))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(ct_arr[best_z],  cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title('CT (axial)')
    axes[0].axis('off')
    axes[1].imshow(det_arr[best_z], cmap='hot',  vmin=0,    vmax=1)
    axes[1].set_title('Detection map')
    axes[1].axis('off')
    axes[2].imshow(ct_arr[best_z],  cmap='gray', vmin=-200, vmax=300)
    axes[2].imshow(det_arr[best_z], cmap='hot',  vmin=0,    vmax=1,  alpha=0.5)
    axes[2].set_title(f'Overlay  |  score = {likelihoods[case_id]:.4f}')
    axes[2].axis('off')
    fig.suptitle(case_id, fontsize=12)
    plt.tight_layout()
    plt.show()

## 14. (Optional) Stage-2 training from scratch

Skip if using the pretrained Stage-2 checkpoints.

**Requirements before running:**
- Section 8 (plan & preprocess) must have completed successfully
- GPU with ≥ 24 GB VRAM recommended

In [ ]:
TRAIN_STAGE2 = False   # set to True to enable

if TRAIN_STAGE2:
    os.environ['nnUNet_results']      = MODEL_DIR
    os.environ['nnUNet_raw']          = NNUNET_RAW
    os.environ['nnUNet_preprocessed'] = NNUNET_PREPROCESSED

    for fold in range(N_FOLDS):
        train_cmd = [
            'nnUNetv2_train',
            str(STAGE2_TASK),
            '3d_fullres',
            str(fold),
            '-tr', STAGE2_TRAINER,
            '-p',  STAGE2_PLAN,
        ]
        print(f'Training fold {fold}:', ' '.join(train_cmd))
        subprocess.check_call(train_cmd)
else:
    print('Training skipped (TRAIN_STAGE2=False). Using pretrained checkpoints.')

## 15. Summary report

In [ ]:
with open(osp.join(OUTPUT_DIR, 'pdac-likelihood.json')) as f:
    scores = json.load(f)

print('=' * 58)
print('  PanDx — patient-level PDAC likelihood scores')
print('=' * 58)
for case, s in sorted(scores.items()):
    bar   = '█' * int(s * 40)
    label = 'HIGH' if s >= 0.5 else 'low '
    print(f'  {case:35s}  {s:.4f}  {label}  {bar}')
print('=' * 58)

vals = list(scores.values())
if vals:
    plt.figure(figsize=(max(4, len(vals) * 1.5), 4))
    plt.bar(scores.keys(), vals,
            color=['tomato' if v >= 0.5 else 'steelblue' for v in vals])
    plt.axhline(0.5, color='red', linestyle='--', label='threshold = 0.5')
    plt.ylim(0, 1)
    plt.ylabel('PDAC likelihood')
    plt.title('Patient-level detection scores')
    plt.xticks(rotation=30, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 16. Cleanup intermediate files

Removes the `itm/` working directory (low-res images, cropped images, raw .npz files).  
Final detection maps and likelihood JSON in `output/` are **preserved**.

In [ ]:
CLEANUP = False   # set to True to free disk space

if CLEANUP and osp.exists(WORKING_DIR):
    shutil.rmtree(WORKING_DIR)
    print(f'Removed: {WORKING_DIR}')
else:
    print(f'Cleanup skipped. Intermediate files at: {WORKING_DIR}')

---
# Part C — Evaluation & Verification
---

## 17. System & hardware logging

Document GPU model, memory, and timing for reproducibility reporting.

In [ ]:
import platform, datetime
import torch

print('=' * 60)
print('  Hardware & Environment Log')
print('=' * 60)
print(f'  Date/Time  : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Python     : {platform.python_version()}')
print(f'  PyTorch    : {torch.__version__}')
print(f'  Platform   : {platform.platform()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'  GPU name   : {gpu.name}')
    print(f'  GPU memory : {gpu.total_memory / 1e9:.2f} GB')
    print(f'  CUDA ver   : {torch.version.cuda}')
    print(f'  GPU count  : {torch.cuda.device_count()}')
else:
    print('  GPU        : NOT available — running on CPU')
print('=' * 60)

hw_log = {
    'date':           datetime.datetime.now().isoformat(),
    'python':         platform.python_version(),
    'pytorch':        torch.__version__,
    'platform':       platform.platform(),
    'gpu_name':       torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU',
    'gpu_memory_gb':  round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 0,
    'cuda_version':   torch.version.cuda if torch.cuda.is_available() else 'N/A',
}
write_json(osp.join(OUTPUT_DIR, 'hardware_log.json'), hw_log)
print(f'Log saved to: {osp.join(OUTPUT_DIR, "hardware_log.json")}')

## 18. Build ground-truth labels for evaluation

Match each case's predicted likelihood score to its ground-truth PDAC label (1 = PDAC+, 0 = PDAC-).

In [ ]:
# Load the predicted likelihood scores
with open(osp.join(OUTPUT_DIR, 'pdac-likelihood.json')) as f:
    likelihoods = json.load(f)

# Load ground truth from the DASE split CSV
split_csv = osp.join(REPO_ROOT, 'workspace', 'dase_split.csv')

if osp.exists(split_csv):
    gt_df = pd.read_csv(split_csv)[['case_id', 'is_pdac']]
else:
    # Build ground truth directly from label masks if CSV not available
    gt_records = []
    for case_id in likelihoods:
        lbl_path = None
        for ext in ['.nii.gz', '.mha', '.nii']:
            p = osp.join(LABEL_DIR, f'{case_id}{ext}')
            if osp.exists(p):
                lbl_path = p
                break
        if lbl_path:
            arr = sitk.GetArrayFromImage(sitk.ReadImage(lbl_path))
            gt_records.append({'case_id': case_id, 'is_pdac': bool((arr == 2).any())})
    gt_df = pd.DataFrame(gt_records)

# Build aligned arrays: predicted score vs ground truth label
eval_df = pd.DataFrame({
    'case_id':    list(likelihoods.keys()),
    'pred_score': list(likelihoods.values()),
}).merge(gt_df, on='case_id', how='inner')

y_true  = eval_df['is_pdac'].astype(int).values
y_score = eval_df['pred_score'].values

print(f'Cases for evaluation : {len(eval_df)}')
print(f'PDAC positive        : {y_true.sum()}')
print(f'PDAC negative        : {(y_true == 0).sum()}')
print(f'Score range          : [{y_score.min():.4f}, {y_score.max():.4f}]')

## 19. Key metric reproduction

**Targets from Liu et al. (2025):**
| Metric | PanDx (paper) | Challenge baseline |
|---|---|---|
| AUROC | **0.9263** | 0.9223 |
| Average Precision | **0.7243** | 0.6335 |

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

# ── Compute metrics ───────────────────────────────────────────────────────────
auroc = roc_auc_score(y_true, y_score)
ap    = average_precision_score(y_true, y_score)

TARGET_AUROC = 0.9263
TARGET_AP    = 0.7243
BASE_AUROC   = 0.9223
BASE_AP      = 0.6335

print('=' * 58)
print('  Metric Reproduction Results')
print('=' * 58)
print(f'  {"Metric":<25} {"Ours":>8} {"Target":>8} {"Baseline":>10} {"Gap":>8}')
print(f'  {"-"*55}')
print(f'  {"AUROC":<25} {auroc:>8.4f} {TARGET_AUROC:>8.4f} {BASE_AUROC:>10.4f} {auroc-TARGET_AUROC:>+8.4f}')
print(f'  {"Average Precision":<25} {ap:>8.4f} {TARGET_AP:>8.4f} {BASE_AP:>10.4f} {ap-TARGET_AP:>+8.4f}')
print('=' * 58)

print()
print('  AUROC: ' + ('REPRODUCED ✅ (within 0.01)' if abs(auroc - TARGET_AUROC) < 0.01 else f'DEVIATION ⚠️  (diff = {auroc-TARGET_AUROC:+.4f})'))
print('  AP   : ' + ('REPRODUCED ✅ (within 0.02)' if abs(ap - TARGET_AP) < 0.02 else f'DEVIATION ⚠️  (diff = {ap-TARGET_AP:+.4f})'))
print('  vs Baseline AUROC: ' + (f'BETTER by {auroc-BASE_AUROC:+.4f} ✅' if auroc > BASE_AUROC else f'WORSE by {auroc-BASE_AUROC:+.4f} ❌'))
print('  vs Baseline AP   : ' + (f'BETTER by {ap-BASE_AP:+.4f} ✅'    if ap > BASE_AP    else f'WORSE by {ap-BASE_AP:+.4f} ❌'))

write_json(osp.join(OUTPUT_DIR, 'evaluation_metrics.json'), {
    'AUROC': round(auroc, 4), 'Average_Precision': round(ap, 4),
    'target_AUROC': TARGET_AUROC, 'target_AP': TARGET_AP,
    'baseline_AUROC': BASE_AUROC, 'baseline_AP': BASE_AP,
    'n_cases': int(len(eval_df)), 'n_pdac_pos': int(y_true.sum()),
})
print(f'\nMetrics saved to: {osp.join(OUTPUT_DIR, "evaluation_metrics.json")}')

In [ ]:
# ── ROC and Precision-Recall curves ──────────────────────────────────────────
fpr, tpr, _   = roc_curve(y_true, y_score)
prec, rec, _  = precision_recall_curve(y_true, y_score)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, color='tomato', lw=2, label=f'PanDx (AUROC = {auroc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curve\nTarget AUROC = {TARGET_AUROC} | Baseline = {BASE_AUROC}')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1])

axes[1].plot(rec, prec, color='steelblue', lw=2, label=f'PanDx (AP = {ap:.4f})')
axes[1].axhline(y_true.mean(), color='k', linestyle='--', lw=1,
                label=f'Random (prevalence = {y_true.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curve\nTarget AP = {TARGET_AP} | Baseline = {BASE_AP}')
axes[1].legend(fontsize=9)
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'roc_pr_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {osp.join(OUTPUT_DIR, "roc_pr_curves.png")}')

## 20. Ablation — DASE vs no-DASE split

**Paper claim:** DASE improves AP from 0.7983 → 0.8247 on validation set. AUROC stays stable.

We verify by comparing per-fold metrics under DASE vs standard random stratified split.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ── Build no-DASE (standard stratified) split ─────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
eval_df['fold_nodase'] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(eval_df, eval_df['is_pdac'])):
    eval_df.loc[eval_df.index[val_idx], 'fold_nodase'] = fold_idx

# Load DASE fold assignments
if osp.exists(split_csv):
    dase_folds = pd.read_csv(split_csv)[['case_id', 'fold']].rename(columns={'fold': 'fold_dase'})
    eval_df = eval_df.merge(dase_folds, on='case_id', how='left')
else:
    eval_df['fold_dase'] = eval_df['fold_nodase']

# ── Per-fold metrics for both splits ─────────────────────────────────────────
results = {'dase': [], 'nodase': []}
for split_name, fold_col in [('dase', 'fold_dase'), ('nodase', 'fold_nodase')]:
    for fold in range(N_FOLDS):
        val = eval_df[eval_df[fold_col] == fold]
        if val['is_pdac'].sum() == 0 or val['is_pdac'].sum() == len(val):
            continue
        results[split_name].append({
            'fold':  fold,
            'auroc': roc_auc_score(val['is_pdac'], val['pred_score']),
            'ap':    average_precision_score(val['is_pdac'], val['pred_score']),
        })

dase_r   = pd.DataFrame(results['dase'])
nodase_r = pd.DataFrame(results['nodase'])

print('=' * 62)
print('  DASE vs No-DASE — per-fold metrics')
print('=' * 62)
print(f'  {"":8} {"DASE AUROC":>12} {"DASE AP":>10} {"Rand AUROC":>12} {"Rand AP":>10}')
print(f'  {"-"*54}')
for f in range(N_FOLDS):
    d = dase_r[dase_r.fold == f]
    n = nodase_r[nodase_r.fold == f]
    if len(d) and len(n):
        print(f'  Fold {f}   {d["auroc"].values[0]:>12.4f} {d["ap"].values[0]:>10.4f} '
              f'{n["auroc"].values[0]:>12.4f} {n["ap"].values[0]:>10.4f}')
print(f'  {"-"*54}')
print(f'  {"Mean":<8} {dase_r["auroc"].mean():>12.4f} {dase_r["ap"].mean():>10.4f} '
      f'{nodase_r["auroc"].mean():>12.4f} {nodase_r["ap"].mean():>10.4f}')
print(f'  {"Std":<8} {dase_r["auroc"].std():>12.4f} {dase_r["ap"].std():>10.4f} '
      f'{nodase_r["auroc"].std():>12.4f} {nodase_r["ap"].std():>10.4f}')
print('=' * 62)

ap_diff = dase_r['ap'].mean() - nodase_r['ap'].mean()
print(f'\n  AP improvement from DASE : {ap_diff:+.4f}')
print(f'  Paper reports            : +0.0264 (0.7983 → 0.8247)')
print('  DASE AP improvement ' + ('CONFIRMED ✅' if ap_diff > 0 else 'NOT confirmed on this subset ⚠️'))

In [ ]:
# ── Visualise DASE vs no-DASE per-fold ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(N_FOLDS);  w = 0.35

for ax, metric, title, target in [
    (axes[0], 'auroc', 'AUROC per fold — DASE vs No-DASE', TARGET_AUROC),
    (axes[1], 'ap',    'AP per fold — DASE vs No-DASE',    TARGET_AP),
]:
    dv = [dase_r[dase_r.fold == f][metric].values[0]   if len(dase_r[dase_r.fold == f])   else 0 for f in range(N_FOLDS)]
    nv = [nodase_r[nodase_r.fold == f][metric].values[0] if len(nodase_r[nodase_r.fold == f]) else 0 for f in range(N_FOLDS)]
    ax.bar(x - w/2, dv, w, label='DASE',    color='tomato',    edgecolor='white')
    ax.bar(x + w/2, nv, w, label='No-DASE', color='steelblue', edgecolor='white')
    ax.axhline(target, color='black', linestyle='--', lw=1.5, label=f'Paper target ({target})')
    ax.set_xticks(x); ax.set_xticklabels([f'Fold {i}' for i in range(N_FOLDS)])
    ax.set_ylabel(metric.upper()); ax.set_title(title)
    ax.legend(fontsize=8); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'dase_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 21. α sensitivity curve (reproduces Fig. 2 of the paper)

Sweep `1/α` from 1 to 30. The paper reports **1/α = 15 is optimal**.
We verify by computing AUROC and AP at each value using the existing probability maps.

In [ ]:
alpha_values = list(range(1, 31))
alpha_aurocs = []
alpha_aps    = []

npz_fps_eval = sorted(glob(osp.join(CROPPED_PRED_DIR, '*.npz')))

print('Sweeping 1/α from 1 to 30...')
for inv_a in tqdm(alpha_values, desc='α sweep'):
    scores_a = {}
    for npz_fp, img_fp in zip(npz_fps_eval, infer_img_paths):
        ext     = get_file_extension(img_fp)
        case_id = osp.basename(npz_fp)[:-4]
        ref_img = sitk.ReadImage(img_fp, sitk.sitkFloat32)
        prob_map = postprocess_probmap(npz_fp)
        _, score = build_full_size_detection_map(
            prob_map, crop_coordinates[case_id], ref_img, inv_alpha=inv_a
        )
        scores_a[case_id] = score

    a_eval = eval_df.copy()
    a_eval['pred_score'] = a_eval['case_id'].map(scores_a)
    a_eval = a_eval.dropna(subset=['pred_score'])

    if a_eval['is_pdac'].sum() > 0:
        alpha_aurocs.append(roc_auc_score(a_eval['is_pdac'], a_eval['pred_score']))
        alpha_aps.append(average_precision_score(a_eval['is_pdac'], a_eval['pred_score']))
    else:
        alpha_aurocs.append(np.nan)
        alpha_aps.append(np.nan)

best_auroc_idx = int(np.nanargmax(alpha_aurocs))
best_ap_idx    = int(np.nanargmax(alpha_aps))
print(f'\nOptimal 1/α for AUROC : {alpha_values[best_auroc_idx]}  (AUROC = {alpha_aurocs[best_auroc_idx]:.4f})')
print(f'Optimal 1/α for AP    : {alpha_values[best_ap_idx]}  (AP    = {alpha_aps[best_ap_idx]:.4f})')
print(f'Paper reports optimal : 1/α = 15')
print('α = 15 is optimal: ' + ('CONFIRMED ✅' if alpha_values[best_ap_idx] == 15 else f'Best found at {alpha_values[best_ap_idx]} ⚠️'))

In [ ]:
# ── Plot α sensitivity curve (Fig. 2 reproduction) ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, vals, metric_name, best_idx, target in [
    (axes[0], alpha_aurocs, 'AUROC', best_auroc_idx, TARGET_AUROC),
    (axes[1], alpha_aps,    'AP',    best_ap_idx,    TARGET_AP),
]:
    ax.plot(alpha_values, vals, 'o-', color='steelblue', lw=2, markersize=5)
    ax.axvline(15, color='red',   linestyle='--', lw=2,   label='Paper optimal (1/α = 15)')
    ax.axvline(alpha_values[best_idx], color='green', linestyle=':',
               lw=1.5, label=f'Our optimal (1/α = {alpha_values[best_idx]})')
    ax.axhline(target, color='gray', linestyle='--', lw=1, label=f'Paper target ({target})')
    ax.set_xlabel('1/α  (inverse alpha)')
    ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name} vs 1/α  —  Fig. 2 reproduction')
    ax.legend(fontsize=8)
    ax.set_xlim([1, 30])

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'alpha_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {osp.join(OUTPUT_DIR, "alpha_sensitivity.png")}')

## 22. Inference timing & GPU memory usage

In [ ]:
import torch

timing_records = []
for npz_fp, img_fp in zip(npz_fps_eval, infer_img_paths):
    ext     = get_file_extension(img_fp)
    case_id = osp.basename(npz_fp)[:-4]

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    ref_img  = sitk.ReadImage(img_fp, sitk.sitkFloat32)
    prob_map = postprocess_probmap(npz_fp)
    _, score = build_full_size_detection_map(
        prob_map, crop_coordinates[case_id], ref_img, INV_ALPHA
    )
    elapsed = time.time() - t0

    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    timing_records.append({
        'case_id':         case_id,
        'postproc_time_s': round(elapsed, 3),
        'peak_gpu_mem_gb': round(peak_mem, 3),
        'likelihood':      round(score, 4),
    })

timing_df = pd.DataFrame(timing_records)
print('Post-processing timing per case:')
print(timing_df.to_string(index=False))
print(f'\nMean post-processing time : {timing_df["postproc_time_s"].mean():.3f}s per case')
print(f'Peak GPU memory           : {timing_df["peak_gpu_mem_gb"].max():.3f} GB')

timing_df.to_csv(osp.join(OUTPUT_DIR, 'timing_log.csv'), index=False)
print(f'Timing log saved to: {osp.join(OUTPUT_DIR, "timing_log.csv")}')

## 23. Final documentation report

Consolidates all results into one report with honest documentation of uncertainty and limitations.

In [ ]:
import datetime

print('=' * 70)
print('  PanDx Reproduction — Final Report')
print(f'  Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 70)

print('\n── 1. Hardware ──────────────────────────────────────────────────────')
with open(osp.join(OUTPUT_DIR, 'hardware_log.json')) as f:
    hw = json.load(f)
print(f'  GPU       : {hw["gpu_name"]}')
print(f'  GPU memory: {hw["gpu_memory_gb"]} GB')
print(f'  CUDA      : {hw["cuda_version"]}')
print(f'  PyTorch   : {hw["pytorch"]}')

print('\n── 2. Key Metrics ───────────────────────────────────────────────────')
auroc_match = '✅' if abs(auroc - TARGET_AUROC) < 0.01 else '⚠️ '
ap_match    = '✅' if abs(ap - TARGET_AP) < 0.02 else '⚠️ '
print(f'  {"Metric":<25} {"Reproduced":>12} {"Paper":>10} {"Baseline":>10} {"Match?":>7}')
print(f'  {"-"*64}')
print(f'  {"AUROC":<25} {auroc:>12.4f} {TARGET_AUROC:>10.4f} {BASE_AUROC:>10.4f} {auroc_match:>7}')
print(f'  {"Average Precision":<25} {ap:>12.4f} {TARGET_AP:>10.4f} {BASE_AP:>10.4f} {ap_match:>7}')

print('\n── 3. DASE Ablation ─────────────────────────────────────────────────')
print(f'  Mean AP with DASE    : {dase_r["ap"].mean():.4f} ± {dase_r["ap"].std():.4f}')
print(f'  Mean AP without DASE : {nodase_r["ap"].mean():.4f} ± {nodase_r["ap"].std():.4f}')
print(f'  AP improvement       : {dase_r["ap"].mean() - nodase_r["ap"].mean():+.4f}  (paper: +0.0264)')

print('\n── 4. α Sensitivity ─────────────────────────────────────────────────')
print(f'  Optimal 1/α (AUROC)  : {alpha_values[best_auroc_idx]}')
print(f'  Optimal 1/α (AP)     : {alpha_values[best_ap_idx]}')
print(f'  Paper reports optimal: 15')

print('\n── 5. Timing ────────────────────────────────────────────────────────')
print(f'  Mean post-processing : {timing_df["postproc_time_s"].mean():.3f}s per case')
print(f'  Peak GPU memory      : {timing_df["peak_gpu_mem_gb"].max():.3f} GB')

print('\n── 6. Limitations & Uncertainty (medical AI honesty) ────────────────')
print('  - Metrics on available subset; full PANORAMA test set may differ')
print('  - nnU-Net TTA introduces minor stochasticity across runs')
print('  - DASE ablation reuses same predictions; only split assignment varies')
print('  - α sweep uses post-processing only; full model retrain per α infeasible')
print('  - Hardware differences may cause small numerical differences vs paper')
print('  - Results must NOT be used for clinical diagnostic decisions')
print('=' * 70)

# Save full JSON report
final_report = {
    'hardware': hw,
    'metrics': {
        'AUROC': round(auroc, 4), 'AP': round(ap, 4),
        'target_AUROC': TARGET_AUROC, 'target_AP': TARGET_AP,
        'baseline_AUROC': BASE_AUROC, 'baseline_AP': BASE_AP,
    },
    'dase_ablation': {
        'mean_AP_dase':   round(dase_r['ap'].mean(), 4),
        'mean_AP_nodase': round(nodase_r['ap'].mean(), 4),
        'ap_improvement': round(dase_r['ap'].mean() - nodase_r['ap'].mean(), 4),
    },
    'alpha_sensitivity': {
        'optimal_inv_alpha_auroc': alpha_values[best_auroc_idx],
        'optimal_inv_alpha_ap':    alpha_values[best_ap_idx],
        'paper_optimal': 15,
    },
    'timing': {
        'mean_postproc_s':  round(timing_df['postproc_time_s'].mean(), 3),
        'peak_gpu_mem_gb':  round(timing_df['peak_gpu_mem_gb'].max(), 3),
    },
}
write_json(osp.join(OUTPUT_DIR, 'final_report.json'), final_report)
print(f'\nFull report saved to: {osp.join(OUTPUT_DIR, "final_report.json")}')